# Cross-Lingual NLI with XLM-RoBERTa: End-to-End Pipeline

Final project for *Introduction to Computational Linguistics* — University of Vienna, Winter 2026.  
**Elisa Lopes de Souza**

This notebook walks through the full pipeline: data loading, four fine-tuning experiments, final test-set evaluation, embedding extraction for visualization, and qualitative error analysis. The heavy lifting lives in the `src/` package so the logic can also be run from the command line via the scripts in `scripts/`.

## 1. Setup

In [1]:
# Colab only
!pip install -q transformers datasets evaluate accelerate tensorboard scikit-learn

In [ ]:
!git clone https://github.com/elisalopes22/xlm-roberta-xnli-cross-lingual.git
%cd xlm-roberta-xnli-cross-lingual

import os, sys

sys.path.insert(0, os.path.abspath(".."))

fatal: destination path 'xlm-roberta-xnli-cross-lingual' already exists and is not an empty directory.
/content/xlm-roberta-xnli-cross-lingual


In [3]:
from src.config import EXPERIMENTS, BEST_EXPERIMENT, LABELS, SEED
from src.data import (
    load_xnli_splits,
    build_mixed_train,
    build_mixed_validation,
    tokenize_splits,
)
from src.train import build_trainer, load_tokenizer, set_seed
from src.evaluate import evaluate_per_language, pretty_print_results
from src.error_analysis import collect_predictions, summarise
from src.embeddings import extract_layer_embeddings

set_seed(SEED)
OUTPUT_ROOT = "./results/checkpoints"

## 2. Load XNLI and build mixed EN+TR splits

Validation is deliberately built by concatenating the EN and TR validation splits so the model is monitored on both languages at every epoch.

In [4]:
tokenizer = load_tokenizer()
xnli_en, xnli_tr = load_xnli_splits()

val_multi = build_mixed_validation(xnli_en, xnli_tr)
print(f"Mixed validation size: {len(val_multi)}")

tokenized_val = tokenize_splits(
    {"en": xnli_en["validation"], "tr": xnli_tr["validation"], "mixed": val_multi},
    tokenizer,
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Mixed validation size: 4980


Map:   0%|          | 0/2490 [00:00<?, ? examples/s]

## 3. Four fine-tuning experiments

Rather than duplicating 30+ lines of `TrainingArguments` four times, each experiment is a declarative `ExperimentConfig` in `src/config.py` and `build_trainer` turns it into a configured `transformers.Trainer`.

| Experiment | Train | Epochs | LR | Weight decay | Purpose |
|---|---|---|---|---|---|
| Exp 1 | 5k | 4 | 2e-5 | 0.01 | Baseline |
| Exp 2 | 5k | 4 | 2e-5 | 0.07 | More regularisation |
| Exp 3 | 5k | 6 | 1e-5 | 0.01 | Slower, longer |
| Exp 4 | 10k | 4 | 2e-5 | 0.01 | **More data (best)** |

In [5]:
trainers = {}

for key, cfg in EXPERIMENTS.items():
    print(f"\n{'=' * 70}\nRunning {cfg.name}\n{'=' * 70}")
    print(f"  {cfg.notes}")

    train_ds = build_mixed_train(xnli_en, xnli_tr, cfg.train_size_per_lang)
    tokenized_train = tokenize_splits({"train": train_ds}, tokenizer)["train"]

    trainer = build_trainer(
        experiment=cfg,
        train_dataset=tokenized_train,
        eval_dataset=tokenized_val["mixed"],
        output_root=OUTPUT_ROOT,
        tokenizer=tokenizer,
    )
    trainer.train()

    results = evaluate_per_language(
        trainer,
        {"EN": tokenized_val["en"], "TR": tokenized_val["tr"], "mixed": tokenized_val["mixed"]},
    )
    pretty_print_results(results, title=f"{cfg.name} validation")
    trainers[key] = trainer


Running Exp1_Baseline
  Standard fine-tuning; serves as reference point.


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

XLMRobertaForSequenceClassification LOAD REPORT from: xlm-roberta-base
Key                         | Status     | 
----------------------------+------------+-
lm_head.layer_norm.weight   | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
classifier.out_proj.weight  | MISSING    | 
classifier.out_proj.bias    | MISSING    | 
classifier.dense.weight     | MISSING    | 
classifier.dense.bias       | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Epoch,Training Loss,Validation Loss,Accuracy
1,1.098996,1.048282,0.423494
2,0.990850,0.930383,0.585542
3,0.828582,0.857761,0.638554
4,0.696677,0.867119,0.648394


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['roberta.embeddings.LayerNorm.weight', 'roberta.embeddings.LayerNorm.bias', 'roberta.encoder.layer.0.attention.output.LayerNorm.weight', 'roberta.encoder.layer.0.attention.output.LayerNorm.bias', 'roberta.encoder.layer.0.output.LayerNorm.weight', 'roberta.encoder.layer.0.output.LayerNorm.bias', 'roberta.encoder.layer.1.attention.output.LayerNorm.weight', 'roberta.encoder.layer.1.attention.output.LayerNorm.bias', 'roberta.encoder.layer.1.output.LayerNorm.weight', 'roberta.encoder.layer.1.output.LayerNorm.bias', 'roberta.encoder.layer.2.attention.output.LayerNorm.weight', 'roberta.encoder.layer.2.attention.output.LayerNorm.bias', 'roberta.encoder.layer.2.output.LayerNorm.weight', 'roberta.encoder.layer.2.output.LayerNorm.bias', 'roberta.encoder.layer.3.attention.output.LayerNorm.weight', 'roberta.encoder.layer.3.attention.output.LayerNorm.bias', 'roberta.encoder.layer.3.output.LayerNorm.weight', 'roberta.encoder.layer.3.output.Laye


Exp4_BigData validation
-----------------------
  EN         0.7233
  TR         0.6341
  mixed      0.6787


## 4. Final evaluation on the held-out XNLI test set

The test set is used exactly once, on the best model selected from validation (Experiment 4).

In [6]:
tokenized_test = tokenize_splits(
    {"en": xnli_en["test"], "tr": xnli_tr["test"]},
    tokenizer,
)

best_trainer = trainers[BEST_EXPERIMENT]

test_results = evaluate_per_language(
    best_trainer,
    {"English (source)": tokenized_test["en"], "Turkish (target)": tokenized_test["tr"]},
)
pretty_print_results(test_results, title="Final test-set accuracy")

gap = test_results["English (source)"] - test_results["Turkish (target)"]
print(f"\nEN -> TR transfer gap: {gap:+.4f}")

Map:   0%|          | 0/5010 [00:00<?, ? examples/s]

Map:   0%|          | 0/5010 [00:00<?, ? examples/s]


Final test-set accuracy
-----------------------
  English (source) 0.7178
  Turkish (target) 0.6555

EN -> TR transfer gap: +0.0623


## 5. Hidden-state embeddings for the TensorBoard Projector

Extract the start-token (`<s>`) representation at three layers (input embeddings (0), middle encoder (6), and final encoder (12)), so the learned semantic geometry can be inspected visually.

For RoBERTa-based models the start token lives at position `[0]`, which is why `src/embeddings.py` indexes positionally instead of searching for a `[CLS]` token ID.

In [7]:
premises = val_multi["premise"][:500]
hypotheses = val_multi["hypothesis"][:500]
labels = val_multi["label"][:500]

extract_layer_embeddings(
    model=best_trainer.model,
    tokenizer=tokenizer,
    premises=premises,
    hypotheses=hypotheses,
    labels=labels,
    output_dir="./results/embeddings",
    layers_of_interest=[0, 6, 12],
    max_examples=500,
)

# The generated TSV files were uploaded to https://projector.tensorflow.org

  saved: Layer_0
  saved: Layer_6
  saved: Layer_12_Final


['./results/embeddings/Layer_0',
 './results/embeddings/Layer_6',
 './results/embeddings/Layer_12_Final']

## 6. Qualitative error analysis

The Turkish error patterns discussed in the report (§5) surface most clearly on the Turkish validation split. The analysis below prints the first handful of errors and successes for each split, the report walks through specific morphological cases in depth.

In [8]:
for name, ds in [
    ("Validation (Turkish)", tokenize_splits({"x": xnli_tr["validation"]}, tokenizer)["x"]),
    ("Test (Turkish)", tokenized_test["tr"]),
    ("Test (English)", tokenized_test["en"]),
]:
    records = collect_predictions(best_trainer, ds)
    summarise(records, dataset_name=name, num_errors=5, num_successes=3)

Map:   0%|          | 0/2490 [00:00<?, ? examples/s]


=== Validation (Turkish) ===
Total: 2490  |  Correct: 1579  |  Errors: 911  |  Accuracy: 0.6341

First 5 errors:
  [0]
    Premise:    Ve Anne, evdeyim dedi.
    Hypothesis: Okul servisi onu bırakır bırakmaz annesini aradı.
    Gold: Neutral  |  Predicted: Contradiction
  [3]
    Premise:    Ne için gittiğimi falan bilmiyordum, Washington'da belirtilen bir yere rapor vermem gerekiyordu.
    Hypothesis: Washington'a hiç gitmedim, bu yüzden oraya atandığımda yeri bulmaya çalışırken kayboldum.
    Gold: Neutral  |  Predicted: Contradiction
  [7]
    Premise:    Gitmesine gerek yoktu.
    Hypothesis: Katılması yasaktı.
    Gold: Entailment  |  Predicted: Contradiction
  [8]
    Premise:    Gitmesine gerek yoktu.
    Hypothesis: Müze açılışına gitmesine izin verilmemişti.
    Gold: Neutral  |  Predicted: Contradiction
  [9]
    Premise:    Benim için sorun yoktu, ve hepsi bu kadardı!
    Hypothesis: Ben evet dedikten sonra sona erdi.
    Gold: Entailment  |  Predicted: Contradiction

First


=== Test (Turkish) ===
Total: 5010  |  Correct: 3284  |  Errors: 1726  |  Accuracy: 0.6555

First 5 errors:
  [0]
    Premise:    Pekala, bunu hiç düşünmemiştim ancak kafam çok karıştı ve onunla tekrar konuşmadım.
    Hypothesis: Onunla bir daha konuşmadım.
    Gold: Contradiction  |  Predicted: Entailment
  [2]
    Premise:    Pekala, bunu hiç düşünmemiştim ancak kafam çok karıştı ve onunla tekrar konuşmadım.
    Hypothesis: Harika bir konuşma yaptık.
    Gold: Neutral  |  Predicted: Contradiction
  [4]
    Premise:    Ve bunun bir ayrıcalık olduğunu sanıyordum, ve hala, hala benim, AFFC Hava Kuvvetleri Kariyer alanım olan dokuz tane iki iki X-O'ydu.
    Hypothesis: AFFC Hava Kuvvetleri Kariyer alanında o numaraya sahip olan tek kişi olduğum izlenimindeydim.
    Gold: Entailment  |  Predicted: Neutral
  [5]
    Premise:    Ve bunun bir ayrıcalık olduğunu sanıyordum, ve hala, hala benim, AFFC Hava Kuvvetleri Kariyer alanım olan dokuz tane iki iki X-O'ydu.
    Hypothesis: Her ne kadar 


=== Test (English) ===
Total: 5010  |  Correct: 3596  |  Errors: 1414  |  Accuracy: 0.7178

First 5 errors:
  [2]
    Premise:    Well, I wasn't even thinking about that, but I was so frustrated, and, I ended up talking to him again.
    Hypothesis: We had a great talk.
    Gold: Neutral  |  Predicted: Contradiction
  [13]
    Premise:    That was the primary thing we wanted to save since there wasn't any way to dump a 20-megaton H-bomb off a 30, a C124.
    Hypothesis: We wanted to save one thing more than the rest.
    Gold: Entailment  |  Predicted: Neutral
  [26]
    Premise:    I was the only one that uh, ever run the regulators for the, the test in the miniature altitude chambers.
    Hypothesis: There were a few of us who ran the regulators for the test.
    Gold: Contradiction  |  Predicted: Entailment
  [34]
    Premise:    The girl that can help me is all the way across town.
    Hypothesis: The girl who is going to help me is 5 miles away.
    Gold: Neutral  |  Predicted: C

## 7. Takeaways

- Training data size was the single biggest lever: 5k → 10k improved validation accuracy by roughly 6 points, more than any regularisation or learning-rate adjustment achieved.
- The EN→TR test gap of 6.15 pp is consistent with XLM-R's known behavior on morphologically distant mid-resource targets under a constrained training budget.
- Errors cluster around cases where lexical overlap is weak — often driven by Turkish agglutination fragmenting roots across subword tokens, or by pragmatic ambiguity in machine-translated premises.